In [ ]:
# ============================================================
# STEP 1: Load the Dataset
# ============================================================
import pandas as pd

df = pd.read_csv('iris.csv')
df

In [ ]:
# ============================================================
# STEP 2: Display the Dataset (First 5 Rows)
# ============================================================
print("--- First 5 Rows of Dataset ---")
print(df.head())

In [ ]:
# ============================================================
# STEP 3: Check Dataset Information
# ============================================================
print("--- Dataset Info ---")
df.info()

In [ ]:
# ============================================================
# STEP 4: Display Dataset Shape
# ============================================================
print(f"Dataset Shape: {df.shape[0]} Rows, {df.shape[1]} Columns")

In [ ]:
# ============================================================
# STEP 5: Display Column Names
# ============================================================
print("Column Names:")
print(list(df.columns))

In [ ]:
# ============================================================
# STEP 6: Display Summary Statistics
# ============================================================
print("--- Summary Statistics ---")
print(df.describe(include='all'))

In [ ]:
# ============================================================
# STEP 7: Check Data Types
# ============================================================
print("Data Types:")
print(df.dtypes)

In [ ]:
# ============================================================
# STEP 8: Check Missing Values
# ============================================================
print("Missing Values Count:")
print(df.isnull().sum())

In [ ]:
# ============================================================
# STEP 9: Handle Missing Values
# ============================================================
import numpy as np

num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(include=['object']).columns

for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Total Missing Values After Handling:", df.isnull().sum().sum())

In [ ]:
# ============================================================
# STEP 10: Remove Duplicate Records
# ============================================================
initial_rows = len(df)
df = df.drop_duplicates()
print(f"Removed {initial_rows - len(df)} duplicate row(s). New shape: {df.shape}")

In [ ]:
# ============================================================
# STEP 11: Encode Categorical Data (Label Encoding)
# ============================================================
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in df.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f"Encoded column '{col}' successfully.")

In [ ]:
# ============================================================
# STEP 12: Rename Columns (lowercase, underscores)
# ============================================================
df = df.rename(columns=lambda x: x.strip().lower().replace(" ", "_"))
print("Updated Column Names:", list(df.columns))

In [ ]:
# ============================================================
# STEP 13: Drop Unnecessary Columns
# ============================================================
unnecessary_cols = ['id', 'order_id', 'customer_id', 'order_date']
drop_targets = [col for col in unnecessary_cols if col in df.columns]

if drop_targets:
    df = df.drop(columns=drop_targets)
    print(f"Dropped columns: {drop_targets}")
else:
    print("No unnecessary columns found to drop.")

In [ ]:
# ============================================================
# STEP 14: Select Features (Feature Selection)
# ============================================================
target_col = 'species'  # Change to 'customer_rating' for ecomer.csv
feature_cols = [col for col in df.columns if col != target_col]

X_selected = df[feature_cols].copy()
print("Selected Features:", list(X_selected.columns))

In [ ]:
# ============================================================
# STEP 15: Create New Features (Feature Engineering)
# ============================================================
if 'sepal_length' in df.columns and 'sepal_width' in df.columns:
    df['sepal_ratio'] = df['sepal_length'] / df['sepal_width']
    df['petal_area'] = df['petal_length'] * df['petal_width']
    print("Created features: 'sepal_ratio' and 'petal_area'")
elif 'unit_price' in df.columns and 'quantity' in df.columns:
    df['effective_unit_price'] = df['unit_price'] * (1 - df['discount'])
    df['quantity_per_delivery_day'] = df['quantity'] / (df['delivery_days'] + 1e-5)
    print("Created features: 'effective_unit_price' and 'quantity_per_delivery_day'")

In [ ]:
# ============================================================
# STEP 16: Convert Data Types (float64 -> float32 for features)
# ============================================================
target_col = 'species'

for col in df.columns:
    if col != target_col and df[col].dtype == 'float64':
        df[col] = df[col].astype('float32')

print("Converted float64 feature columns to float32.")

In [ ]:
# ============================================================
# STEP 17: Split Features and Target Variable
# ============================================================
target_col = 'species'  # Change to 'customer_rating' for ecomer.csv

X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

print(f"Features (X) Shape: {X.shape} | Target (y) Shape: {y.shape}")

In [ ]:
# ============================================================
# STEP 18: Normalize Data (Min-Max Scaling -> 0..1)
# ============================================================
from sklearn.preprocessing import MinMaxScaler

min_max_scaler = MinMaxScaler()
X_normalized = pd.DataFrame(min_max_scaler.fit_transform(X), columns=X.columns)

print("--- Normalized Features (First 2 Rows) ---")
print(X_normalized.head(2))

In [ ]:
# ============================================================
# STEP 19: Standardize Data (Standard Scaling -> Mean=0, Var=1)
# ============================================================
from sklearn.preprocessing import StandardScaler

std_scaler = StandardScaler()
X_standardized = pd.DataFrame(std_scaler.fit_transform(X), columns=X.columns)

print("--- Standardized Features (First 2 Rows) ---")
print(X_standardized.head(2))

In [ ]:
# ============================================================
# STEP 20: Split Dataset into Training and Testing Sets (80/20)
# ============================================================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_standardized, y, test_size=0.2, random_state=42,
    stratify=y if len(np.unique(y)) > 1 else None
)

print(f"X_train Shape: {X_train.shape} | X_test Shape: {X_test.shape}")

In [ ]:
# ============================================================
# STEP 21: Train Classifier and Display Accuracies
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

classifier = RandomForestClassifier(random_state=42)
classifier.fit(X_train, y_train)

train_acc = accuracy_score(y_train, classifier.predict(X_train))
test_acc = accuracy_score(y_test, classifier.predict(X_test))

print(f"Training Accuracy : {train_acc * 100:.2f}%")
print(f"Testing Accuracy  : {test_acc * 100:.2f}%")